# AgriVision AI — 05 Model Export to ONNX

Converts PyTorch checkpoint weights into standardized ONNX graph format for edge acceleration and cross-platform deployment.

In [ ]:
import sys
from pathlib import Path
import torch

project_root = Path.cwd().resolve().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ai.engines.plant_vision.classifier import PlantDiseaseClassifier
from ai.engines.plant_vision.config import PlantVisionConfig
from ai.config import EXPORTS_DIR

print(f"Export destination: {EXPORTS_DIR}")

## 1. Export PyTorch Model to ONNX

In [ ]:
cfg = PlantVisionConfig(architecture="mobilenetv3_large_100", pretrained=False)
classifier = PlantDiseaseClassifier(config=cfg, device="cpu")
classifier.model.eval()

dummy_input = torch.randn(1, 3, 224, 224, requires_grad=False)
export_path = EXPORTS_DIR / "mobilenetv3_disease.onnx"

torch.onnx.export(
    classifier.model,
    dummy_input,
    str(export_path),
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},
)

print(f"Successfully exported ONNX model to: {export_path}")
print(f"File size: {export_path.stat().st_size / (1024 * 1024):.2f} MB")